# Generate the end to end gold set

## Setup

In [ ]:
import os, json, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

PROJECT_ROOT     = Path(os.getcwd()).parent
PARSED_CLEAN_DIR = PROJECT_ROOT / "data" / "parsed_clean"
EVAL_DIR         = PROJECT_ROOT / "data" / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
GOLD_PATH        = EVAL_DIR / "gold_set.json"

load_dotenv(PROJECT_ROOT / ".env", override=True)
client = OpenAI(base_url=os.getenv("GATEWAY_URL", ""), api_key=os.getenv("BEARER_TOKEN", ""))

GOLDSET_MODEL = os.getenv("VL_MODEL_GATEWAY", "")          
TUTOR_MODEL   = os.getenv("INFERENCE_MODEL_GATEWAY", "")   
assert GOLDSET_MODEL, "VL_MODEL_GATEWAY not set"
print("Gold-set author  :", GOLDSET_MODEL)
print("System under test:", TUTOR_MODEL)

## Load the cleaned corpus

Loading the cleaned corpus and grouping the chunks by lecture, the index reused by the multi-hop, global and unanswerable generators

In [ ]:
def load_chunks() -> list[dict]:
    chunks = []
    for file in sorted(PARSED_CLEAN_DIR.rglob("*.json")):
        chunks.extend(json.loads(file.read_text(encoding="utf-8")))
    return chunks

chunks = load_chunks()
by_id  = {c["id"]: c for c in chunks}

by_lecture: dict[str, list[dict]] = {}
for c in chunks:
    by_lecture.setdefault(c["lecture"], []).append(c)

print(f"{len(chunks)} chunks geladen aus {PARSED_CLEAN_DIR}")
print(f"{len(by_lecture)} Vorlesungseinheiten")

## Generate single-hop items

Defining the generation schema, the standalone prompt, the JSON call helper and a deterministic audit guard that auto-flags context leakage and ungrounded quotes

In [ ]:
import re, unicodedata

class GoldDraft(BaseModel):
    question: str = Field(default="")
    reference_answer: str = Field(default="")
    answer_quote: str = Field(default="")

GOLD_SYSTEM_PROMPT = """Du erstellst Evaluationsfragen (Gold-Set) für ein RAG-System, das Fragen zu Vorlesungsunterlagen des Moduls "Machine Learning" beantwortet.

# EINGABE
Genau EIN Auszug (Titel + Inhalt) aus den Unterlagen. Es handelt sich um Vorlesungsfolien oder Auszüge aus Jupyter-Notebooks.

# AUFGABE
Formuliere GENAU EINE Frage, die ein Bachelorstudent realistisch an ein Tutor-/RAG-System stellen würde, während er mit diesem Stoff lernt – plus eine knappe, fachlich korrekte Referenzantwort, die sich vollständig und eindeutig allein aus diesem Auszug beantworten lässt.
Zweck: Getestet wird, ob das RAG-System den relevanten Auszug findet und die Information korrekt wiedergibt. Die Frage muss daher inhaltlich eindeutig auf die im Auszug enthaltene Information zielen – aber ohne den Auszug als Quelle zu erwähnen (siehe Regeln).

# FRAGESTIL (Realismus + Varianz)
Schreibe so, wie ein Student wirklich tippt: kurz und direkt, nicht wie eine Klausurfrage. Decke über das Gold-Set hinweg ein natürliches Spektrum ab – wähle pro Auszug die EINE Formulierung, die ein Student zu genau diesem Inhalt am ehesten stellen würde:
- Niedrigschwellige Verständnisfragen, z. B. "Was bedeutet Entropie bei einem Entscheidungsbaum?", "Wofür braucht man Slack-Variablen bei SVMs?".
- Präzisere Fachfragen, z. B. "Wann konvergiert das Perzeptron-Lernverfahren garantiert?", "Worin unterscheiden sich Gini-Koeffizient und Informationsgewinn?".
- Typische Fragewörter: "Was ist/bedeutet ...?", "Warum ...?", "Wofür ...?", "Wann ...?", "Wie funktioniert ...?", "Was ist der Unterschied zwischen ... und ...?".
Richte Stil und Tiefe nach dem Inhalt: Definitionen laden zu "Was ist ...?" ein, Verfahren/Vergleiche zu "Warum/Wann/Unterschied".
VERMEIDE verschachtelte Mehrfachfragen im Klausurstil (z. B. "..., und was bedeutet ..., und wie ...?"). Höchstens zwei eng zusammenhängende Aspekte, und nur wenn ein Student sie natürlich zusammen fragen würde. Keine künstlich akademische Sprache und keine Aufzählung aller Details des Auszugs – eine echte, fokussierte Frage.

# HARTE REGELN (standalone)
1. STANDALONE: Die Frage muss für sich allein verständlich sein. Jemand, der den Auszug NICHT vor sich hat, muss eindeutig wissen, worum es geht. Nenne das konkrete Konzept beim Namen (z. B. "beim Perzeptron-Lernverfahren", "bei Support Vector Machines", "bei der L2-Regularisierung").
2. KEINE QUELLEN-/DARSTELLUNGSVERWEISE. Verboten sind u. a.: "im Auszug", "im Text", "in diesem Abschnitt", "auf der Folie", "im gegebenen/dargestellten Beispiel", "in der gezeigten/dargestellten Grafik", "in der Abbildung", "hier", "oben", "unten", "wie gezeigt", "laut Folie".
3. KONZEPT STATT DARSTELLUNG: Frage nicht nach dem Aussehen einer Grafik (Farben, Pfeile, Achsen, Linienart). Frage nach dem ML-Konzept, das die Grafik, Formel oder der Code vermittelt.
4. KEINE Ja/Nein-Frage und kein wörtliches Abfragen eines einzelnen Satzes.
5. EINDEUTIG ABLEITBAR: Die Frage darf NUR mit Information aus genau diesem Auszug korrekt beantwortbar sein. Verlange kein Vorwissen und keine Information, die im Auszug fehlt.

# UNGEEIGNETE AUSZÜGE
Enthält der Auszug keine inhaltlich prüfbare Aussage (reine Titelfolie, Inhaltsverzeichnis, Gliederung, Literaturliste, Code ohne erklärenden Kontext, organisatorische Hinweise), dann erfinde KEINE Frage. Gib stattdessen exakt aus:
{"question": "", "reference_answer": "", "answer_quote": ""}

# REFERENZANTWORT
- Nur Information aus dem Auszug, keine externe Ergänzung.
- 1–3 Sätze, fachlich präzise, in eigenen Worten zusammengefasst (kein bloßes Kopieren).
- Selbst ebenfalls standalone formuliert.

# answer_quote
- Ein WÖRTLICHES Zitat aus dem Auszug-Inhalt, das die Referenzantwort belegt.
- Muss ein EXAKTER, zeichengenauer Substring des Auszugs sein (gleiche Schreibweise, Zeichen, Formeln). Erfinde, kürze oder normalisiere nichts.
- So kurz wie möglich, aber ausreichend, um die Antwort zu stützen.

# AUSGABEFORMAT
Antworte AUSSCHLIESSLICH mit einem einzigen JSON-Objekt, ohne Markdown, ohne Einleitung:
{"question": "...", "reference_answer": "...", "answer_quote": "..."}

# BEISPIELE

## Beispiel A – niedrigschwellige Verständnisfrage
Auszug:
"Aufteilungsstrategie Informationsgewinn: Die Entropie misst die Unreinheit (impurity) einer Menge; niedriger bedeutet reiner. Sie ist 0, wenn alle Instanzen zur selben Klasse gehören."
Ausgabe:
{"question": "Was bedeutet die Entropie bei einem Entscheidungsbaum?", "reference_answer": "Die Entropie misst die Unreinheit einer Menge von Trainingsinstanzen bezüglich der Klassen; ein niedriger Wert bedeutet, dass die Menge reiner ist, und sie ist 0, wenn alle Instanzen zur selben Klasse gehören.", "answer_quote": "Die Entropie misst die Unreinheit (impurity) einer Menge; niedriger bedeutet reiner"}

## Beispiel B – präzisere Fachfrage
Auszug:
"Perzeptron-Lernregel: Bei jeder Fehlklassifikation wird der Gewichtsvektor um η·(y - ŷ)·x angepasst. Das Verfahren konvergiert garantiert, wenn die Daten linear separierbar sind."
Ausgabe:
{"question": "Wann konvergiert das Perzeptron-Lernverfahren garantiert?", "reference_answer": "Das Perzeptron konvergiert garantiert, sofern die Trainingsdaten linear separierbar sind; bei jeder Fehlklassifikation wird der Gewichtsvektor um η·(y - ŷ)·x angepasst.", "answer_quote": "Das Verfahren konvergiert garantiert, wenn die Daten linear separierbar sind."}

"""


def passage(c: dict) -> str:
    return "\n\n".join(x for x in [c.get("title"), c.get("page_content")] if x)

def chat_json(system: str, user: str, temperature: float = 0, max_tokens: int = 1024) -> dict:
    resp = client.chat.completions.create(
        model=GOLDSET_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}, 
        temperature=temperature, max_tokens=max_tokens,
    )
    content = resp.choices[0].message.content
    if not content:
        raise ValueError("empty model response (thinking still on?)")
    return json.loads(content)

def make_gold(chunk: dict) -> GoldDraft:
    data = chat_json(GOLD_SYSTEM_PROMPT, f"AUSZUG:\n{passage(chunk)}", max_tokens=2048)
    return GoldDraft.model_validate(data)

Generating single-hop items: one standalone question per sampled chunk, each auto-audited so leaked or ungrounded drafts are marked keep=False

In [ ]:
SINGLE_SIZE = 5  
MIN_CHARS   = 100

random.seed(40)
pool = [c for c in chunks if len(c.get("page_content", "")) >= MIN_CHARS]
if SINGLE_SIZE:
    pool = random.sample(pool, min(SINGLE_SIZE, len(pool)))

single_items = []
for i, c in enumerate(pool, 1):
    try:
        d = make_gold(c)
    except Exception as e:
        print(f"[err  ] {c['id']}: {e}")
        continue
    single_items.append({
        "question":         d.question.strip(),
        "reference_answer": d.reference_answer.strip(),
        "answer_quote":     d.answer_quote.strip(),
        "source_chunk_id":  c["id"],
        "lecture":          c["lecture"],
        "type":             "single_hop",
    })

print(f"\nsingle_hop: {len(single_items)} items generated")

GOLD_PATH.write_text(json.dumps(single_items, indent=2, ensure_ascii=False), encoding="utf-8")  